# DA6401 — Section 2.7: Novel Image Pipeline Showcase

Run 3 novel pet images (from the internet, NOT from the dataset) through the
full pipeline: Classification + Localization + Segmentation. Log to W&B.

**BEFORE RUNNING:**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **ON**
3. Click **Save Version -> Save & Run All (Commit)**

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch, numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import gdown, re

CLASSIFIER_ID = "1x_qgIKPFwu_2inuIjyms7IhQY2Qr_ZKP"
LOCALIZER_ID  = "1jRM83Xr6HETheUhv1MQqRMEl2eAwbkSK"
UNET_ID       = "1ATvVvuMPagLhP3wLjI-sWe3D1Z4wHV0Z"

for did, name in [(CLASSIFIER_ID,'classifier.pth'),(LOCALIZER_ID,'localizer.pth'),(UNET_ID,'unet.pth')]:
    out = f"{CKPT}/{name}"
    gdown.download(id=did, output=out, quiet=True, fuzzy=True)
    print(f"  {name}: {os.path.getsize(out)/1e6:.0f} MB")

In [ ]:
def _canonicalize_checkpoint(sd):
    """Normalize nested-block key format to our flat convention."""
    if not any(re.match(r'encoder\.block\d+\.\d+\.\d+\.', k) for k in sd):
        return sd
    new_sd = {}
    for k, v in sd.items():
        m = re.match(r'encoder\.block(\d+)\.(\d+)\.(\d+)\.(.*)', k)
        if m:
            N, M, L, suf = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
            X = N - 1
            if X in (0, 1):
                Y = 0 if L == 0 else 1
            else:
                Y = (0 if L == 0 else 1) if M == 0 else (3 if L == 0 else 4)
            new_sd[f'encoder.block{X}.{Y}.{suf}'] = v
        elif k.startswith('classifier.'):
            new_sd['head.' + k[len('classifier.'):]] = v
        elif k.startswith('regressor.'):
            new_sd['head.' + k[len('regressor.'):]] = v
    return new_sd

from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet

cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
loc_model = VGG11Localizer(image_size=224).to(device)
seg_model = VGG11UNet(num_classes=3).to(device)

cls_model.load_state_dict(
    _canonicalize_checkpoint(torch.load(f'{CKPT}/classifier.pth', map_location=device, weights_only=False)),
    strict=False)
loc_model.load_state_dict(
    _canonicalize_checkpoint(torch.load(f'{CKPT}/localizer.pth', map_location=device, weights_only=False)),
    strict=False)
seg_model.load_state_dict(
    torch.load(f'{CKPT}/unet.pth', map_location=device, weights_only=False),
    strict=False)

cls_model.eval(); loc_model.eval(); seg_model.eval()
print('All 3 models loaded.')

In [ ]:
import gdown
from PIL import Image as PILImage

NOVEL_DIR = '/kaggle/working/novel_images'
os.makedirs(NOVEL_DIR, exist_ok=True)

NOVEL_IMAGES = [
    ('golden_retriever.jpg', '1-BZEN9e8Vu3ROu1x9KaQ57Ur31tkEbJP'),
    ('pug_dog.jpg',          '1_XkCoH460hbYgMVyqjUFYCfQlxGUwsMh'),
    ('tabby_cat.jpg',        '1-okjVBueJiiVilL7qEdUd47VrqsrweF-'),
]

novel_paths = []
for fname, drive_id in NOVEL_IMAGES:
    out_path = f'{NOVEL_DIR}/{fname}'
    gdown.download(id=drive_id, output=out_path, quiet=True, fuzzy=True)
    img = PILImage.open(out_path).convert('RGB')
    print(f'  {fname}: {img.size[0]}x{img.size[1]}')
    novel_paths.append(out_path)

print(f'\n{len(novel_paths)} novel images ready.')

In [ ]:
import wandb
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from data.pets_dataset import OxfordIIITPetDataset, IMAGENET_MEAN, IMAGENET_STD
from train import denormalize_image

IMAGE_SIZE = 224
class_names = OxfordIIITPetDataset.CLASS_NAMES

# Trimap colormap: 0=foreground(green), 1=background(dark), 2=boundary(yellow)
colormap = np.array([[0, 200, 0], [40, 40, 40], [255, 255, 0]], dtype=np.uint8)

transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

wandb.init(project='da6401-assignment2',
           name='section_2.7_novel_images',
           tags=['exp-novel', 'report'])

result_images = []

for img_path in novel_paths:
    fname = os.path.basename(img_path)
    orig_img = np.array(PILImage.open(img_path).convert('RGB'))
    transformed = transform(image=orig_img)
    img_tensor = transformed['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        # Classification
        cls_logits = cls_model(img_tensor)
        cls_probs = torch.softmax(cls_logits, dim=1)
        conf, pred_idx = cls_probs.max(dim=1)
        breed = class_names[pred_idx.item()]
        confidence = conf.item()

        # Top-3 predictions
        top3_vals, top3_idx = cls_probs.topk(3, dim=1)
        top3_str = ', '.join([f'{class_names[i]}({v:.2f})'
                              for v, i in zip(top3_vals[0].tolist(), top3_idx[0].tolist())])

        # Localization
        bbox = loc_model(img_tensor).cpu().numpy()[0]  # [cx, cy, w, h]

        # Segmentation
        seg_mask = seg_model(img_tensor).argmax(dim=1).cpu().numpy()[0]

    # ---- 4-panel visualization ----
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    resized = np.array(PILImage.fromarray(orig_img).resize((IMAGE_SIZE, IMAGE_SIZE)))

    # Panel 1: Original
    axes[0].imshow(resized)
    axes[0].set_title('Input Image', fontsize=12)
    axes[0].axis('off')

    # Panel 2: Classification
    axes[1].imshow(resized)
    axes[1].set_title(f'Predicted: {breed}\nConfidence: {confidence:.3f}', fontsize=11,
                      color='green' if confidence > 0.5 else 'red')
    axes[1].axis('off')

    # Panel 3: Bounding box
    img_denorm = denormalize_image(img_tensor[0])
    axes[2].imshow(img_denorm)
    x1 = bbox[0] - bbox[2] / 2
    y1 = bbox[1] - bbox[3] / 2
    rect = patches.Rectangle((x1, y1), bbox[2], bbox[3],
                              linewidth=3, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].set_title('Bounding Box', fontsize=12)
    axes[2].axis('off')

    # Panel 4: Segmentation overlay
    mask_colored = colormap[seg_mask]
    overlay = (0.5 * resized.astype(float) + 0.5 * mask_colored.astype(float)).astype(np.uint8)
    axes[3].imshow(overlay)
    axes[3].set_title('Segmentation Overlay', fontsize=12)
    axes[3].axis('off')

    plt.suptitle(f'{fname}\nTop-3: {top3_str}', fontsize=12, fontweight='bold')
    plt.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)

    result_images.append(wandb.Image(
        PILImage.open(buf),
        caption=f'{fname}: {breed} ({confidence:.2f})'))

    print(f'  {fname}: {breed} ({confidence:.3f})')
    print(f'    bbox: cx={bbox[0]:.0f} cy={bbox[1]:.0f} w={bbox[2]:.0f} h={bbox[3]:.0f}')
    print(f'    top-3: {top3_str}')
    print()

wandb.log({'novel_images/pipeline_output': result_images})
wandb.finish()
print('Section 2.7 done — logged to W&B as novel_images/pipeline_output')